# Demostración en Vivo
Generador de Reportes de Partidos de Fútbol — Grupo 9

Esta notebook está pensada para la exposición: permite buscar partidos por equipo
y generar la crónica correspondiente en tiempo real.

In [ ]:
import pandas as pd
from groq import Groq
from dotenv import load_dotenv
import os

load_dotenv()
client = Groq(api_key=os.getenv("GROQ_API_KEY"))

df = pd.read_csv("../data/processed/epl_clean.csv")
print(f"Dataset listo: {df.shape[0]} partidos disponibles")

In [ ]:
def build_prompt(row, estilo="estandar"):
    resultado_map = {'H': f"ganó {row['HomeTeam']}", 'A': f"ganó {row['AwayTeam']}", 'D': "empate"}
    resultado_ht_map = {'H': f"ganaba {row['HomeTeam']}", 'A': f"ganaba {row['AwayTeam']}", 'D': "iban empatados"}
    sorpresa = "Sí, fue una sorpresa" if row['Upset'] else "No, ganó el favorito o empató"

    roles = {
        "estandar": "Eres un periodista deportivo experto en fútbol inglés. Usá un tono dinámico y profesional.",
        "tecnico": "Eres un analista de estadísticas de fútbol. Usá un tono objetivo y analítico, centrado en números.",
        "dramatico": "Eres un comentarista deportivo apasionado. Usá metáforas, suspenso y lenguaje vívido."
    }

    datos = f"""
--- DATOS DEL PARTIDO ---
Fecha: {row['MatchDate']}
Local: {row['HomeTeam']}
Visitante: {row['AwayTeam']}
Resultado final: {int(row['FTHome'])} - {int(row['FTAway'])} ({resultado_map[row['FTResult']]})
Al descanso: {int(row['HTHome'])} - {int(row['HTAway'])} ({resultado_ht_map[row['HTResult']]})

Estadísticas:
- Tiros (local / visitante): {int(row['HomeShots'])} / {int(row['AwayShots'])}
- Tiros al arco (local / visitante): {int(row['HomeTarget'])} / {int(row['AwayTarget'])}
- Corners (local / visitante): {int(row['HomeCorners'])} / {int(row['AwayCorners'])}
- Faltas (local / visitante): {int(row['HomeFouls'])} / {int(row['AwayFouls'])}
- Tarjetas amarillas (local / visitante): {int(row['HomeYellow'])} / {int(row['AwayYellow'])}
- Tarjetas rojas (local / visitante): {int(row['HomeRed'])} / {int(row['AwayRed'])}

Contexto:
- Favorito según cuotas: {row['Favorite']}
- ¿Fue sorpresa el resultado?: {sorpresa}
- Forma reciente local (últ. 5): {row['Form5Home']} pts
- Forma reciente visitante (últ. 5): {row['Form5Away']} pts
- Diferencia de Elo: {row['EloDiff']} puntos
"""

    return f"""{roles[estilo]}
Redactá una crónica periodística en español de aproximadamente 200 palabras sobre el siguiente partido de la Premier League.
La crónica debe tener un título atractivo y destacar los datos más relevantes. Acordate que si el nombre del equipo es Man significa Manchester, no lo acortes. Evitá repetir los nombres de los equipos en exceso y no uses abreviaturas. No incluyas información que no esté en los datos proporcionados.
{datos}"""


def generate_cronica(prompt):
    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[{"role": "user", "content": prompt}],
        max_tokens=500
    )
    return response.choices[0].message.content


def buscar_partidos(equipo):
    resultados = df[
        (df['HomeTeam'].str.contains(equipo, case=False, na=False)) |
        (df['AwayTeam'].str.contains(equipo, case=False, na=False))
    ][['MatchDate', 'HomeTeam', 'AwayTeam', 'FTHome', 'FTAway']].reset_index()
    return resultados

print("Funciones listas")

## Paso 1: Buscar partidos por equipo

In [ ]:
equipo_buscado = "Liverpool"

resultados = buscar_partidos(equipo_buscado)
print(f"Se encontraron {len(resultados)} partidos\n")
resultados.tail(20)

## Paso 2: Elegir el partido por su índice

In [ ]:
indice_elegido = 0

partido = df.loc[resultados.loc[indice_elegido, 'index']]
print(f"Partido seleccionado: {partido['HomeTeam']} vs {partido['AwayTeam']} ({partido['MatchDate']})")
print(f"Resultado: {int(partido['FTHome'])} - {int(partido['FTAway'])}")

## Paso 3: Generar la crónica

In [ ]:
prompt = build_prompt(partido, estilo="estandar")
cronica = generate_cronica(prompt)

print("=" * 60)
print(f"  {partido['HomeTeam']} vs {partido['AwayTeam']}")
print("=" * 60)
print(cronica)

## Bonus: comparar estilos para el mismo partido

In [ ]:
for estilo in ["estandar", "tecnico", "dramatico"]:
    print(f"\n{'='*20} ESTILO: {estilo.upper()} {'='*20}\n")
    cronica_estilo = generate_cronica(build_prompt(partido, estilo=estilo))
    print(cronica_estilo)